In [1]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import torch

In [2]:
import yaml
with open("/home/samuelebumbaca/repositories/Paper2/agri-downstream/src/object_detection/config/experiment_config/config_YOLOv8_SAGIT22_C_4175_2022_05_27.yaml", 'r') as stream:
    try:
        config = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

In [3]:
model_path = '/home/samuelebumbaca/repositories/Paper2/agri-downstream/experiments/YOLOv8_SAGIT22_C_4175_2022_05_27/20250129_122933/weights/best.pt'
image_path = "test/dataset_1.tif"

In [ ]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

detection_model = AutoDetectionModel.from_pretrained(
    model_type= 'ultralytics' if 'yolo' in config['train']['model'] or 'rtdetr' in config['train']['model'] else 'torchvision',
    model_path=model_path,
    confidence_threshold=0.3,
    device=device,
)

Using device: cuda


In [5]:
result = get_sliced_prediction(
    image_path,
    detection_model,
    slice_height = 224,
    slice_width = 224,
    overlap_height_ratio = 0.5,
    overlap_width_ratio = 0.5
)

Performing prediction on 37665 slices.


In [6]:
result.export_visuals(export_dir="dataset_1", 
                      hide_conf=False, 
                      hide_labels=False,
                      text_size=.3,)

In [7]:
boxes = [obj.bbox.to_xyxy() for obj in result.object_prediction_list]
scores = [obj.score.value for obj in result.object_prediction_list]

In [8]:
import rasterio
from shapely.geometry import Point
from shapely.affinity import affine_transform
import os


with rasterio.open(image_path) as src:
    minx, miny, maxx, maxy = src.bounds
    image_height, image_width = src.height, src.width
tile_width = maxx - minx
tile_height = maxy - miny
scale_x = tile_width / image_width
scale_y = tile_height / image_height
transformation_matrix = [scale_x, 0, 0, -scale_y, minx, maxy]
geo_boxes = []
for box in boxes:
    xmin, ymin, xmax, ymax = box
    points = [(xmin, ymin), (xmax, ymax)]
    transformed_points = []
    for x, y in points:
        point = Point(x, y)
        transformed_point = affine_transform(point, transformation_matrix)
        transformed_points.append((transformed_point.x, transformed_point.y))
    geo_boxes.append(transformed_points)


In [9]:
import fiona
from shapely.geometry import mapping, box as shapely_box

output_file_path = "test/dataset_1.shp"

with rasterio.open(image_path) as src:
    crs = src.crs
schema = {
    'geometry': 'Polygon',
    'properties': {'score': 'float',
                    }
}

with fiona.open(output_file_path, 'w', driver='ESRI Shapefile', crs=crs, schema=schema) as shp:
    for box, score in zip(geo_boxes, scores):
        minx, miny = box[0]
        maxx, maxy = box[1]
        geom = shapely_box(minx, miny, maxx, maxy)
        shp.write({
            'geometry': mapping(geom),
            'properties': {'score': score,
                            }
        })